# SMX quickstart: synthetic spectra

This notebook follows [`examples/quickstart.py`](https://github.com/joseviniciusr/SMX/blob/main/examples/quickstart.py):
generate two classes of spectra, train a classifier, cut the spectral axis into zones and explain the classifier with the
**Spectral Model eXplainer**: a ranking of the zones, the predicate graph behind it, threshold spectra and a faithfulness check.

Everything runs in your browser with [Pyodide](https://pyodide.org). Your data never leaves your machine.
Change how the synthetic spectra are generated, or load your own CSV, then press **Run all**.

In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from smx import SMX, building_spectral_zones, generate_synthetic_spectral_data, plot_threshold_spectrum

# rich output in the Sandbox; plain Jupyter output elsewhere
from smx_sandbox import display, show_dataset, show_faithfulness, show_graph, show_metrics, zone_editor

warnings.filterwarnings("ignore", category=FutureWarning)  # e.g. scikit-learn deprecating SVC(probability=True)

## Parameters

By default the data are synthetic: each class is a sum of Gaussian peaks with random height and width plus noise, on a
spectral axis from 1 to 1000. Class A has a peak at 700 that class B lacks, and class B one at 50 that class A lacks;
their other peaks are shared. Move, add or remove peaks to change what separates the classes.

To use your own spectra, load a CSV with one row per spectrum: spectral columns named by their position (wavelength,
energy, channel, …) and a column with the class. SMX explains one class at a time (the probability of that class
against the rest), so pick the class to explain if there are more than two.

The SMX settings are a lighter version of the quickstart's (3 repetitions of 8 bags instead of 4 of 10) to keep the
browser run short.

In [ ]:
dataset = "synthetic"  # @param {type:"file", label:"Dataset"}
# @markdown **Synthetic spectra** (used when the dataset is `synthetic`)
samples_per_class = 100  # @param {type:"slider", min:20, max:300, step:10, label:"Spectra per class"}
peaks_A = "250, 380, 550, 700, 850"  # @param {type:"string", label:"Class A peak positions"}
peaks_B = "50, 250, 380, 550, 850"  # @param {type:"string", label:"Class B peak positions"}
height_A = 1.0  # @param {type:"number", label:"Class A peak height"}
height_B = 1.4  # @param {type:"number", label:"Class B peak height"}
peak_width = 15  # @param {type:"slider", min:3, max:60, label:"Peak width (σ)"}
noise = 0.04  # @param {type:"number", label:"Noise (std)"}
n_points = 250  # @param {type:"slider", min:100, max:1000, step:50, label:"Points on the axis"}
data_seed = 0  # @param {type:"integer", label:"Data seed"}
# @markdown **Your CSV**
target_column = ""  # @param {type:"column", label:"Class column", placeholder:"Class, or detected"}
positive_class = ""  # @param {type:"string", label:"Class to explain", placeholder:"first class"}
first_column_is_index = False  # @param {type:"boolean", label:"First column is a row index"}
# @markdown **Model and zones**
model_name = "SVC"  # @param ["SVC", "LogisticRegression", "RandomForestClassifier", "MLPClassifier"]
test_size = 0.3  # @param {type:"slider", min:0.1, max:0.5, step:0.05, label:"Test fraction"}
zones = "detect peaks"  # @param ["detect peaks", "equal width", "manual"]
peak_prominence = 0.2  # @param {type:"slider", min:0.02, max:1, step:0.02, label:"Peak prominence (share of range)"}
n_equal_zones = 10  # @param {type:"slider", min:2, max:40, label:"Equal-width zones"}
manual_zones = "1-100, 200-300, 330-430, 500-600, 660-750, 815-890"  # @param {type:"string", label:"Manual zones (start-end, …), or draw them on the chart"}
# @markdown **SMX**
quantiles = "0.25, 0.5, 0.75"  # @param {type:"string", label:"Quantiles"}
n_repetitions = 3  # @param {type:"slider", min:1, max:10, label:"Repetitions"}
n_bags = 8  # @param {type:"slider", min:2, max:30, label:"Bags per repetition"}
n_samples_fraction = 0.8  # @param {type:"slider", min:0.3, max:1, step:0.05, label:"Samples per bag (fraction)"}
perturbation_metric = "probability_shift"  # @param ["probability_shift", "accuracy_drop", "f1_drop"]
perturbation_mode = "median"  # @param ["median", "mean", "constant", "min", "max"]
var_exp = True  # @param {type:"boolean", label:"Weight edges by explained variance"}

## Load the spectra

Spectral columns are the ones whose name is a number; they are sorted along the axis, and missing values are filled with
the column mean. Other columns, such as sample ids, are ignored, except the class column: `Class` if there is one,
otherwise the non-numeric column with the fewest distinct values. With more than two classes, every class but the
explained one becomes `not <class>`.

In [ ]:
def numbers(text):
    """'250, 380 550' -> [250.0, 380.0, 550.0]"""
    return [float(v) for v in re.findall(r"-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(text))]


def is_number(name):
    try:
        float(name)
        return True
    except ValueError:
        return False


if dataset == "synthetic":
    df = generate_synthetic_spectral_data([
        {"name": "A", "n_samples": samples_per_class, "peaks": numbers(peaks_A), "amplitude_mean": height_A,
         "amplitude_std": 0.3, "width_mean": peak_width, "noise_std": noise},
        {"name": "B", "n_samples": samples_per_class, "peaks": numbers(peaks_B), "amplitude_mean": height_B,
         "amplitude_std": 0.4, "width_mean": peak_width, "noise_std": noise},
    ], n_points=n_points, x_min=1, x_max=1000, seed=data_seed)
else:
    df = pd.read_csv(dataset, index_col=0 if first_column_is_index else None)

# the class column: "Class" (as the generator names it), else the non-numeric column with the fewest values
labels = [c for c in df.columns if not is_number(c)]
target = target_column or ("Class" if "Class" in df.columns else min(labels, key=lambda c: df[c].nunique(), default=df.columns[-1]))
df = df.dropna(subset=[target])
spectral = sorted((c for c in df.columns if c != target and is_number(c)), key=float)
if len(spectral) < 10:
    raise ValueError(f"Found {len(spectral)} spectral columns: name each spectral column by its position, e.g. 400, 402.5, …")
X = df[spectral].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean()).reset_index(drop=True)
y = df[target].astype(str).reset_index(drop=True)

classes = sorted(y.unique())
positive = positive_class or classes[0]
if positive not in classes:
    raise ValueError(f"Class to explain {positive!r} is not one of {classes}")
negative = next(c for c in classes if c != positive) if len(classes) == 2 else f"not {positive}"
y = y.where(y == positive, negative)

show_dataset(X, y, positive=positive)

## Train a classifier

A stratified calibration/test split, mean-centred with the calibration mean as in the quickstart. SMX treats the model as
a black box: it only calls `predict_proba`, so any classifier works.

In [ ]:
X_cal, X_test, y_cal, y_test = (part.reset_index(drop=True) for part in train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=42))
X_mean = X_cal.mean()
X_cal_prep, X_test_prep = X_cal - X_mean, X_test - X_mean

MODELS = {
    "SVC": lambda: SVC(kernel="rbf", probability=True, random_state=42),
    "LogisticRegression": lambda: LogisticRegression(max_iter=2000),
    "RandomForestClassifier": lambda: RandomForestClassifier(n_estimators=100, random_state=42),
    "MLPClassifier": lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=42),
}
model = MODELS[model_name]().fit(X_cal_prep, y_cal)
pos = list(model.classes_).index(positive)

y_hat = model.predict(X_test_prep)
show_metrics({
    "Test accuracy": accuracy_score(y_test, y_hat),
    "Test F1 (weighted)": f1_score(y_test, y_hat, average="weighted"),
    "Test ROC AUC": roc_auc_score(y_test == positive, model.predict_proba(X_test_prep)[:, pos]),
})

## Spectral zones

SMX explains the model in terms of zones of the spectral axis, normally chosen by an expert (an element's emission
line, an absorption band, …). Here they are either **detected** on the mean spectrum with `building_spectral_zones`
(peaks become `zone…`, the stretches between them `background…`), cut into **equal-width** zones, or set
**manually**.

To set them by hand, edit them on the chart below: drag across the spectrum to add a zone, drag a zone or its edges
to move it, remove zones with ×. Each edit switches the zone mode to manual and writes the ranges into the form;
press **Run all** to explain the model with them.

In [ ]:
mean_spectrum = X_cal.mean()
if zones == "detect peaks":
    span = float(mean_spectrum.max() - mean_spectrum.min())
    spectral_cuts = building_spectral_zones(mean_spectrum, prominence=peak_prominence * span)
elif zones == "equal width":
    edges = np.linspace(float(X.columns[0]), float(X.columns[-1]), n_equal_zones + 1)
    spectral_cuts = [(f"Z{i + 1}", float(a), float(b)) for i, (a, b) in enumerate(zip(edges[:-1], edges[1:]))]
else:
    pairs = re.findall(r"(-?\d+(?:\.\d+)?)\s*[-–]\s*(-?\d+(?:\.\d+)?)", manual_zones)
    ranges = sorted(sorted((float(a), float(b))) for a, b in pairs)
    spectral_cuts = [(f"Z{i + 1}", a, b) for i, (a, b) in enumerate(ranges)]

print(f"{len(spectral_cuts)} zones:", ", ".join(f"{name} ({a:g}–{b:g})" for name, a, b in spectral_cuts))
zone_editor(mean_spectrum, spectral_cuts)
axis = X.columns.astype(float)
empty = [f"{name} ({a:g}–{b:g})" for name, a, b in spectral_cuts if not ((axis >= a) & (axis <= b)).any()]
if empty:
    raise ValueError(f"No spectral points in {', '.join(empty)}: the axis runs from {axis.min():g} to {axis.max():g}")
if len(spectral_cuts) < 2:
    raise ValueError(f"Got {len(spectral_cuts)} zone(s), SMX needs at least two: draw more on the chart, "
                     "lower the peak prominence or pick another zone mode")

## Explain the model with SMX

Each zone is reduced to its first principal component, and quantiles of that score give predicates such as
`zone3 > 1.2`. In every bag, a random subset of the spectra, SMX perturbs the zone behind each predicate and measures
how much the predicted probability moves; ordering the predicates by that impact gives a path through them. The paths of
all bags and repetitions form a weighted, directed graph, and the **Local Reaching Centrality** (LRC) of its nodes ranks
the predicates and their zones.

This is the slow step: around a minute in the browser with the default settings. The log shows each bag as it runs.

In [ ]:
smx = SMX(
    spectral_cuts=spectral_cuts,
    quantiles=numbers(quantiles),
    n_repetitions=n_repetitions,
    n_bags=n_bags,
    n_samples_fraction=n_samples_fraction,
    estimator=model,
    perturbation_metric=perturbation_metric,
    perturbation_mode=perturbation_mode,
    var_exp=var_exp,
)
# the continuous output SMX explains: the model's probability of the explained class
y_pred_cal = pd.Series(model.predict_proba(X_cal_prep)[:, pos])
smx.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)

display(smx.plot_zone_ranking_over_spectrum(ranking="unique", X_natural=X_cal, y_labels=y_cal, height=420))
ranking = smx.lrc_natural_.dropna(subset=["Zone"]).drop_duplicates("Zone").reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.rename_axis("Rank")[["Zone", "Node", "Local_Reaching_Centrality", "Node_Natural"]]

## The predicate graph

The graph of one repetition. Each node is a predicate on a zone's PC1 score; an edge means the source ranked just above
the target in a bag, weighted by the source's perturbation impact. Paths end in the class the bag's samples were
predicted as. Hover a node to trace its paths, click it for its thresholds and centrality. Try
`show_graph(smx, repetition=1)` for another repetition.

In [ ]:
show_graph(smx, class_names={"A": positive, "B": negative})

## Threshold spectra

A predicate's threshold lives in PCA space, but PCA is invertible: SMX maps it back to a spectrum in the original
units. Here are the top predicates of the two best-ranked zones, drawn over the calibration spectra of each class.

In [ ]:
lrc = smx.lrc_natural_.reset_index(drop=True)
for row in lrc[lrc["Zone"].notna()].drop_duplicates("Zone").index[:2]:
    display(plot_threshold_spectrum(lrc, row, smx.zones_natural_, smx.pca_info_natural_, y_cal, height=380))

## Faithfulness

Does the ranking reflect what the model uses? SMX masks the test spectra zone by zone, best-ranked first, and records
how far the predictions move. The area under that curve is compared with the areas of random zone orders: a faithful
ranking moves the predictions sooner than random ones.

In [ ]:
faithfulness = smx.evaluate_faithfulness(X_test_prep, ranking="unique", masking_strategy="zero", n_random_rankings=50)
show_faithfulness(faithfulness)
smx.plot_faithfulness(height=420)

## Your turn

Add a cell below to explore further, for example `smx.lrc_summed_` (every predicate), `smx.pca_info_` (variance each
zone's PC1 explains), or more of SMX's plots such as `plot_lrc_bar(smx.lrc_natural_)` and
`plot_predicate_heatmap(smx.lrc_natural_)` (import them from `smx`). Plotly figures and matplotlib plots are shown inline.